# Deep Hedging AI: Comparison to Black & Scholes

In this notebook we run Deep Hedging in a Black & Scholes world
_without statistical drift_. In this case we can compare hedging a vanilla call vs its perfect hedge.

### Proof that Deep Hedging should converge to the Black & Scholes price

We are using the notation from the mark down file https://github.com/hansbuehler/deephedging.
Let $Z_T$ be the claim in $T$. Assume that there are no trading cost and that the market is _complete_. This applies to Black & Scholes, and also to most other models typically used in quant finance.
In the limit of $dt\downarrow 0$ that menas there is a perfect replication strateg $\alpha^*$ such that
$$
    (*) \ \ \ \ Z_T \equiv z_0 + \sum_{t=0}^{T-1} \alpha_t^*(s_t)\ ( H_T - H_t)
$$
The constant $z_0$ is the risk-neutral price of $Z_T$.

Consider now our risk-adjusted return metric $U$ as used in Deep Hedging. We are trying to compute
$$
    \sup_a:\ U\left[\ Z_T + \sum_{t=0}^{T-1} a_t(s_t)\ ( H_T - H_t)\ \right]
$$
We now insert $(*)$ for $Z_T$. That gives
$$
    \sup_a:\ U\left[\ z_0 + \sum_{t=0}^{T-1} \alpha_t^*(s_t)\ ( H_T - H_t) + \sum_{t=0}^{T-1} a_t(s_t)\ ( H_T - H_t)\ \right]
$$
Because of _cash-invariance_ we can move $z_0$ out of $U$.
$$
(**) \ \ \ \    z_0 + \sup_a:\ U\left[\ \sum_{t=0}^{T-1} \left( \alpha_t^*(s_t) - a_t(s_t) \right)\ ( H_T - H_t) \ \right]
$$
In case that the underlying path generator is a risk-neutral measure we get $\mathrm{E}_t[ H_T - H_t ] = 0$ and therefore that $\mathrm{E}_t[ x_t(a_t) ( H_T - H_t)  ]=0$ for any possible strategy $x$.
Moreover, because $U$ has the property that $U(X) \leq E[X]$ we see that an optimal solutio for $(**)$ is achived in
$$
    a_t = -\alpha^*_t
$$
which means that the optimal solution to the Deep Hedging problem for _any_ monetary utility is the risk-neutral strategy.
We also note that the utility of $Z_T$ under the Deep Hedging is
$
z_0 + U(0) = z_0 
$
in other words it returns the risk-neutral price as well.

### Questions

* *Why are the hedges for vanilla options not limited by -1/+1:* the hedging signal from Deep Hedging is an interpolation. In the grapsh below we show both the hedge per one of the 10,000 path, and a binned interpolation. In order to increase precision, a bigger network needs to be trained. It is important to notice that the noise does not impede the quality of the hedge when measures via the utility $U$.
* *How does that compare to the results in the Deep Hedging paper:* in that paper we artificially restricted delta to -1/+1. This should have been noted in the model.

In [ ]:
""" Slighly annoying: by default the SageMaker Python import directory does not include our git directory """
#!pip -q install cdxbasics "tensorflow>=2.11" "tensorflow_probability==0.19"
import os
p = os.getcwd()
dhn = "/deephedging/"
i = p.find(dhn)
my_path = None
if i!=-1:
    my_path = p[:i]
    import sys
    sys.path.append(my_path)
    print("SageMaker: added python path %s" % my_path)
else:
    print(sys.path)

In [ ]:
print("Deep Hedging AI says hello  ... ", end='')
from cdxbasics.config import Config
from deephedging.trainer import train
from deephedging.gym import VanillaDeepHedgingGym
from deephedging.world import SimpleWorld_Spot_ATM

from IPython.display import display, Markdown

# see print of the config below for numerous options
config = Config()
# world
config.world.samples = 10000
config.world.steps = 20       # 20 weekly steps
config.world.dt = 5./250.     # """
config.world.drift = 0.       # no drift
config.world.cost_s = 0.        # no cost
config.world.black_scholes = True
# gym
config.gym.objective.utility = "cvar"
config.gym.objective.lmbda = 1.
config.gym.agent.network.depth = 3
config.gym.agent.network.activation = "softplus"
# trainer
config.trainer.train.optimizer.name = "adam"
#config.trainer.train.optimizer.learning_rate = 0.001
#config.trainer.train.optimizer.clipvalue = 0.5
#config.trainer.train.optimizer.global_clipnorm = 1.
config.trainer.train.batch_size = 2000
config.trainer.train.epochs = 20000
config.trainer.caching.mode = "on"
config.trainer.caching.epoch_freq = 100
config.trainer.visual.epoch_refresh = 20
config.trainer.visual.confidence_pcnt_lo = 0.25
config.trainer.visual.confidence_pcnt_hi = 0.75

config_small_dt = config.copy()
config_small_dt.world.steps = 20*5  # 100 daily steps
config_small_dt.world.dt = 1./250.  # """

In [ ]:
display(Markdown("## Deep Hedging in a Black \& Scholes World - Big Steps"))

# create world
world  = SimpleWorld_Spot_ATM( config.world )
val_world  = world.clone(samples=world.nSamples//2)

# create training environment
gym = VanillaDeepHedgingGym( config.gym )

# create training environment
train( gym=gym, world=world, val_world=val_world, config=config.trainer )
config.done()

In [ ]:
import deephedging.plot_bs_hedge as plot_bs_hedge
plot_bs_hedge.plot_blackscholes( world, gym, config )

In [ ]:
display(Markdown("## Deep Hedging in a Black \& Scholes World - Small Steps (this blew up before 26000)"))

config_small_dt.trainer.caching.mode = "on"

# create world
world      = SimpleWorld_Spot_ATM( config_small_dt.world )
val_world  = world.clone(samples=world.nSamples//2)

# create training environment
gym = VanillaDeepHedgingGym( config_small_dt.gym )

# create training environment
train( gym=gym, world=world, val_world=val_world, config=config_small_dt.trainer )
config_small_dt.done()

In [ ]:
import deephedging.plot_bs_hedge as plot_bs_hedge
plot_bs_hedge.plot_blackscholes( world, gym, config_small_dt )